# 03 · Cross-lingual retrieval and the RAG chatbot
The policy is English; questions are English, Tagalog or Taglish. This notebook measures which retrieval signals find the right page, then runs the full chatbot.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_colwidth", 120)
from src.retriever import HybridRetriever, bank_frame
from src.glossary import expand_query
qa = pd.read_csv(config.QA_FILE)

## 1. Glossary expansion

In [2]:
for q in ["Nawala ng airline ang bagahe ko", "Na-delay yung flight ko ng 10 oras", "Sagot ba kung buntis ako?"]:
    print(q, "\n   ->", expand_query(q))

Nawala ng airline ang bagahe ko 
   -> Nawala ng airline ang bagahe ko || lost loss baggage luggage
Na-delay yung flight ko ng 10 oras 
   -> Na-delay yung flight ko ng 10 oras || delay delayed flight
Sagot ba kung buntis ako? 
   -> Sagot ba kung buntis ako? || covered cover pregnancy


## 2. Ablation: Hit@k of the gold page (unseen test questions)
Bank = training items only.

In [3]:
CONFIGS = {"tfidf": dict(lexical="tfidf", use_glossary=False, use_bank=False),
           "bm25": dict(lexical="bm25", use_glossary=False, use_bank=False),
           "bm25+glossary": dict(lexical="bm25", use_glossary=True, use_bank=False),
           "bm25+bank": dict(lexical="bm25", use_glossary=False, use_bank=True),
           "full": dict(lexical="bm25", use_glossary=True, use_bank=True)}
test = qa[qa.split == "test"]
bank = bank_frame(splits=("train",))
rows = []
for name, kw in CONFIGS.items():
    ret = HybridRetriever(qa=bank, **kw)
    for r in test.itertuples():
        gold = {int(p) for p in str(r.pages).split(";") if p not in ("", "nan")}
        if gold:
            top = ret.top_pages(r.input, 5)
            rows.append({"config": name, "language": r.language, "hit@1": bool(gold & set(top[:1])),
                         "hit@3": bool(gold & set(top[:3])), "hit@5": bool(gold & set(top))})
res = pd.DataFrame(rows)
res.groupby(["config", "language"])["hit@3"].mean().unstack().loc[list(CONFIGS)].round(2)

language,english,tagalog,taglish
config,,,
tfidf,0.67,0.50,0.72
bm25,0.61,0.56,0.78
bm25+glossary,0.56,0.83,0.78
bm25+bank,0.56,0.67,0.78
full,0.56,0.89,0.83


In [4]:
res.groupby("config")[["hit@1", "hit@3", "hit@5"]].mean().loc[list(CONFIGS)].round(2)

,hit@1,hit@3,hit@5
config,,,
tfidf,0.35,0.63,0.67
bm25,0.44,0.65,0.72
bm25+glossary,0.52,0.72,0.87
bm25+bank,0.41,0.67,0.80
full,0.52,0.76,0.85


The glossary is what rescues Tagalog questions (Hit@3 0.56 → 0.89). The full results, including the FAQ-paraphrase scenario, are produced by `scripts/evaluate.py` → `results/metrics.json` and `results/charts/`.

## 3. The chatbot
Without `ANTHROPIC_API_KEY` it runs in offline mode (checked reference answers). With a key, Claude writes a grounded answer citing *[Section, p.]* in the customer's language.

In [5]:
from src.chatbot import PolicyChatbot
bot = PolicyChatbot()
for q in ["Nawala ng airline yung maleta ko, ano pwede ko i-claim?",
          "My flight was delayed 10 hours, what can I get?",
          "Sagot ba ng insurance kung buntis ako at nanganak sa Japan?",
          "Ano ang magandang kainan sa Tokyo?"]:
    r = bot.answer(q)
    print(f"Q: {q}\n[{r.mode} | {r.language} | {r.topic} {r.topic_confidence:.0%} | pages {[c['page'] for c in r.chunks]}]\n{r.answer}\n")

Q: Nawala ng airline yung maleta ko, ano pwede ko i-claim?
[offline | taglish | baggage 81% | pages [26, 29, 27, 29, 27]]
Ito yung sinasabi ng policy (offline mode, pinakamalapit na sagot):

Ang babayaran ng Company ay yung portion na hindi covered ng airline liability limit, up to your Policy Schedule limit. Considered lost ang luggage after ng period ng carrier, minimum 21 days. Kailangan ng list of contents with estimated price at date of purchase, at yung settlement ng airline. Excluded ang valuables, cash, jewelry, cards at documents (Section 14, pp. 26-27).

Q: My flight was delayed 10 hours, what can I get?
[offline | english | flight_delay 65% | pages [25, 11, 23, 26, 25]]
Here is what the policy says (offline mode, closest reference answer):

The Company pays for each full 8-hour period the first part of your booked flight is delayed, up to your plan limit, as long as you eventually take the trip. Covered causes are a strike unknown to you, acts of nature, technical failure of

In [6]:
# what Claude actually receives (system prompt + grounded user turn)
p = bot.prepare("Covered ba ako pag nag-scuba diving ako sa Palawan?")
print(p["system"][:900], "...\n")
print(p["messages"][-1]["content"][:1500])

You are "Shield", the Travel Shield Policy Explainer: a friendly assistant that explains Prudential Guarantee's Travel Shield Insurance Policy (and its COVID-19 Endorsement) to Filipino travellers and helps them start a claim.

LANGUAGE
- Detected language of the customer's message: taglish.
- Reply in that same register. english -> clear English. tagalog -> natural, polite Tagalog (use "po" sparingly). taglish -> everyday Filipino code-switching, the way the customer wrote, keeping insurance terms in English (e.g. "covered", "claim", "Policy Schedule").

GROUNDING RULES
1. Use ONLY the facts in <policy_excerpts> and <reference_answers>. The reference answers were written and checked against the policy; prefer them when they match the question, but adapt them to what was actually asked.
2. Cite every fact with its section and page in brackets, e.g. [Section 17, p. 28]. Never cite a page  ...

<policy_excerpts>
<excerpt page="6" section="Definitions">
e rapid spread of infectious diseas

## 4. End-to-end LLM evaluation (needs API key)
`python scripts/eval_llm.py` answers the 57 test questions with Claude and scores citation validity, gold-page citation, language match and a Claude-as-judge correctness score against the reference answers. Results: `results/llm_metrics.json`.

In [ ]:
m = config.RESULTS / "llm_metrics.json"
print(m.read_text() if m.exists() else "Not run yet: set ANTHROPIC_API_KEY and run python scripts/eval_llm.py")

Not run yet: set ANTHROPIC_API_KEY and run python scripts/eval_llm.py


: 